<a href="https://colab.research.google.com/github/Yashground/omnicare-clinical-copilot/blob/main/notebooks/00_setup_and_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniCare Clinical Copilot - Notebook 0: Setup & Models

This notebook sets up the environment, installs dependencies, loads the AI models,
and defines shared utilities used across all other notebooks.

**Models:**
- `google/medgemma-1.5-4b-it` — Medical multimodal LLM (4-bit quantized)
- `openai/whisper-large-v3-turbo` — Speech-to-text ASR

**GPU Budget:** ~5-6 GB on T4 16GB (comfortable headroom)

## 1. Install Dependencies

In [1]:
!pip install -q transformers accelerate bitsandbytes torch
!pip install -q soundfile librosa
!pip install -q pydicom Pillow
!pip install -q fhir.resources pydantic
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 19.2 MB/s eta 0:00:00


## 2. Authenticate with HuggingFace

MedGemma is a gated model — you must have accepted the license at
[huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it).

In [3]:
from huggingface_hub import login

# This will prompt for your HuggingFace token
# You can also set HF_TOKEN environment variable or use Colab secrets
login()

## 3. Check GPU Availability

In [6]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Models will run very slowly on CPU.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
VRAM used: 0.00 GB


## 4. Load MedGemma 1.5-4b-it (4-bit Quantized)

This model handles:
- SOAP note generation from transcripts
- Admission note generation
- Radiology image analysis (multimodal)
- Discharge summary generation

In [8]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"

# 4-bit quantization config — fits in T4 16GB GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {MEDGEMMA_MODEL_ID} with 4-bit quantization...")
medgemma_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)

print(f"MedGemma loaded. VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

Loading google/medgemma-1.5-4b-it with 4-bit quantization...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

MedGemma loaded. VRAM used: 3.44 GB


## 5. Load Whisper Large v3 Turbo (ASR)

Used for medical audio transcription with vocabulary prompting.

In [10]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WHISPER_MODEL_ID = "openai/whisper-large-v3-turbo"

print(f"Loading {WHISPER_MODEL_ID}...")
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID,
    dtype=torch.float16, # Changed torch_dtype to dtype
    low_cpu_mem_usage=True
).to("cuda")

whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL_ID)

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    dtype=torch.float16, # Changed torch_dtype to dtype
    device="cuda"
)

print(f"Whisper loaded. Total VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

Loading openai/whisper-large-v3-turbo...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Whisper loaded. Total VRAM used: 6.68 GB


## 6. Sanity Check — Test Both Models

In [12]:
# Test MedGemma with a simple medical question
test_messages = [
    {"role": "user", "content": [{"type": "text", "text": "What are the common symptoms of pneumonia?"}]}
]

inputs = medgemma_processor.apply_chat_template(
    test_messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(medgemma_model.device)

with torch.no_grad():
    output_ids = medgemma_model.generate(**inputs, max_new_tokens=200)

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
response = medgemma_processor.decode(new_tokens, skip_special_tokens=True)
print("MedGemma test response:")
print(response[:500])

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


MedGemma test response:
Pneumonia is an infection that inflames the air sacs in one or both lungs. The air sacs may fill with fluid or pus, causing a variety of symptoms. Here are some of the most common symptoms of pneumonia:

* **Cough:** This is often the first symptom. It may be dry or produce mucus (phlegm).
* **Fever:** A high fever is common, but not always present.
* **Chills:** Feeling cold and shivering.
* **Shortness of breath:** Difficulty breathing, especially when lying down.
* **Chest pain:** Often sharp


In [13]:
# Test Whisper with a short audio sample (if available)
# For now, just verify the pipeline is functional
print("Whisper ASR pipeline ready.")
print(f"Model: {WHISPER_MODEL_ID}")
print(f"Supports: WAV, MP3, FLAC, OGG audio formats")
print(f"Medical prompting enabled via initial_prompt parameter")

Whisper ASR pipeline ready.
Model: openai/whisper-large-v3-turbo
Supports: WAV, MP3, FLAC, OGG audio formats
Medical prompting enabled via initial_prompt parameter


## 7. GPU Memory Summary

In [14]:
!nvidia-smi

Wed Mar 25 13:28:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             26W /   70W |    6735MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 8. Create Encounter State Directory

In [15]:
import os
import sys

# Add utils to path so notebooks can import shared modules
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# Create encounters directory
os.makedirs("/content/encounters", exist_ok=True)
os.makedirs("/content/sample_data", exist_ok=True)

print("Encounter state directory: /content/encounters/")
print("Sample data directory: /content/sample_data/")
print("\nSetup complete! Proceed to the next notebook.")

Encounter state directory: /content/encounters/
Sample data directory: /content/sample_data/

Setup complete! Proceed to the next notebook.


---

**Next:** Run `01_consultation_audio_soap.ipynb` for the audio transcription and SOAP generation pipeline.

**Important:** Keep this notebook's runtime active — the models are loaded in GPU memory
and shared across notebooks via Colab's session state.